# Imports

In [1]:
%reload_ext autoreload
%autoreload 2


from stnd.utility.data_utils import make_or_load_from_cache
from stnd.utility.utils import apply_random_seed
import sys
import os
import sklearn
import numpy as np
import pandas as pd
from datasets import load_dataset

ROOT_PATH = os.path.join(os.path.dirname(os.path.abspath("")))
sys.path.insert(0, ROOT_PATH)
from experiments import (
    RANDOM_SEED,
    make_train_test_model_embeddings,
    make_cache_subpath,
    make_disagreement_scores_dict,
    make_fitted_weights
)
from utils import (
    lb_scenarios,
    dump_pickle,
    load_pickle,
    prepare_and_split_data
)
from plots import (
    MODEL_OUTPUTS_PATH,
    load_scores,
    safe_spearmanr
)
from selection import sample_items
from run_experiment import load_and_split_model_outputs
from acc import (
    compute_true_acc,
    compute_acc_knn
)
from scripts.evaluate_mmlu import evaluate_mmlu
sys.path.pop(0);


CACHE_DIR = "./cache_dir"

/home/oh/arubinstein17/github/disco-public/envs/disco_env_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Functions

In [2]:
def compute_accuracy(df, metric_key="acc"):
    # Compute accuracy as number of rows where df["metrics"]["acc"] == 1
    # Each element in df["metrics"] is likely a dict with an 'acc' key
    if "metrics" in df.columns:
        num_correct_acc = sum(1 for metric in df["metrics"] if metric.get('acc') == 1)
        num_correct_norm_acc = sum(1 for metric in df["metrics"] if metric.get('norm_acc') == 1)

    else:
        num_correct_acc = sum(1 for acc in df["acc"] if acc == 1)
        num_correct_norm_acc = sum(1 for acc in df["acc_norm"] if acc == 1)

    print(f"Accuracy (number of acc == 1): {num_correct_acc} / {len(df)}")
    print(f"Accuracy (number of acc == 1): {num_correct_acc / len(df)}")
    print(f"Accuracy (number of norm_acc == 1): {num_correct_norm_acc} / {len(df)}")
    print(f"Accuracy (number of norm_acc == 1): {num_correct_norm_acc / len(df)}")


import pandas as pd

# Convert aux['latest'] to a DataFrame and print as a table.
# Try to structure the data so that each example is a row.

def read_per_model_info(model_id, timestamp, scenario, cache_dir=CACHE_DIR):

    # model_id = "meta-llama/Llama-2-13b-hf"
    creator, model = tuple(model_id.split("/"))
    model_details_id = "open-llm-leaderboard/details_{:}__{:}".format(creator, model)

    # s = "harness_hendrycksTest_abstract_algebra_5"
    s = scenario
    aux = load_dataset(model_details_id, s, cache_dir=cache_dir)
    print("Available timestamps:")
    print(aux.keys())

    # Attempt to create a table with all available columns.
    latest_data = aux[timestamp]

    # The structure may be a dict with lists as values, or list of dicts.
    # We need to check what we have.

    if isinstance(latest_data, dict):
        # If values are lists of the same length, treat as column-wise.
        lens = [len(v) for v in latest_data.values() if isinstance(v, list)]
        if lens and all(l == lens[0] for l in lens):
            # Looks like column-wise dict of lists.
            df = pd.DataFrame(latest_data)
        else:
            # Dict where each key might be a scalar or something else.
            df = pd.DataFrame([latest_data])
    else:
        # Try to coerce into DataFrame anyway
        df = pd.DataFrame(latest_data)

    # print(df)
    return df


# Evaluate new model

## Get ground truth

### Take from leaderboard snapshot

In [15]:
df = pd.read_csv(os.path.join(ROOT_PATH, "benchmark_csvs","open-llm-leaderboard.csv"))
# small_models = df[df['#Params (B)'] <= 8]
# print(small_models.head(n=10))
big_models = df[(df['#Params (B)'] >= 70) & (df["T"] == "🟢")]
big_models.head(n=10)


,T,Model,Average ⬆️,ARC,HellaSwag,MMLU,TruthfulQA,Winogrande,GSM8K,Type,...,Weight type,Precision,Merged,Hub License,#Params (B),Hub ❤️,Available on the hub,Model sha,Flagged,MoE
122,🟢,Qwen/Qwen-72B,73.60,65.19,85.94,77.37,60.19,82.48,70.43,pretrained,...,Original,bfloat16,False,other,72.29,253.0,True,f62c59844a8de3c27cf22735218d77e9fa9f6b17,False,False
502,🟢,tiiuae/falcon-180B,67.85,69.45,88.86,70.50,45.47,86.90,45.94,pretrained,...,Original,8bit,False,unknown,179.52,1007.0,True,71a1a70b629e9963f7b4601e82f3f9079d48011e,False,False
654,🟢,tiiuae/falcon-180B,65.46,69.20,88.89,69.59,45.16,86.74,33.21,pretrained,...,Original,4bit,False,unknown,179.52,1007.0,True,71a1a70b629e9963f7b4601e82f3f9079d48011e,False,False
2136,🟢,bigscience/bloom,46.07,50.43,76.41,30.85,39.76,72.06,6.90,pretrained,...,Original,float16,False,bigscience-bloom-rail-1.0,176.25,4304.0,True,053d9cd9fbe814e091294f67fcfedb3397b954bb,False,False


In [10]:
df[df["T"] == "🟢"]

,T,Model,Average ⬆️,ARC,HellaSwag,MMLU,TruthfulQA,Winogrande,GSM8K,Type,...,Weight type,Precision,Merged,Hub License,#Params (B),Hub ❤️,Available on the hub,Model sha,Flagged,MoE
1,🟢,cloudyu/Yi-34Bx2-MoE-60B,76.72,71.08,85.23,77.47,66.19,84.85,75.51,pretrained,...,Original,bfloat16,False,cc-by-nc-4.0,60.81,42.0,True,483359d70b3fef480cdaeb6d722a18626d34f0ce,True,True
2,🟢,cloudyu/Mixtral_34Bx2_MoE_60B,76.66,71.33,85.25,77.34,66.59,84.85,74.60,pretrained,...,Original,bfloat16,False,cc-by-nc-4.0,60.81,83.0,True,f49d7cf0a7b99b15bc98b0ef4a681e7f0f4aa92c,True,True
3,🟢,cloudyu/Mixtral_34Bx2_MoE_60B,76.63,71.25,85.36,77.28,66.61,84.69,74.60,pretrained,...,Original,float16,False,cc-by-nc-4.0,60.81,83.0,True,f49d7cf0a7b99b15bc98b0ef4a681e7f0f4aa92c,True,True
6,🟢,sumo43/Yi-32b-x2-v2.0,76.17,73.04,85.95,76.79,73.22,82.79,65.20,pretrained,...,Original,bfloat16,True,mit,60.81,0.0,True,1e61f28b326fe0080ad476ce2b1dd041ec9f147f,True,True
50,🟢,gagan3012/MetaModel_moe,74.42,71.25,88.40,66.26,71.86,83.35,65.43,pretrained,...,Original,float16,True,apache-2.0,36.10,0.0,True,015dae67b68e6e5007b7b13a448886eb5f6bfea8,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2971,🟢,Locutusque/TinyMistral-248m,27.73,22.87,28.02,23.15,42.52,49.80,0.00,pretrained,...,Original,float16,False,?,0.25,0.0,True,8f03f72bca0542aa164c29ba41f02cba6f9d7748,False,False
2972,🟢,ai-forever/mGPT,27.61,23.81,26.37,25.17,39.62,50.67,0.00,pretrained,...,Original,float16,False,apache-2.0,0.00,202.0,True,40897bd7c8b47a76802c411108ca6220438b8b40,False,False
2979,🟢,FabbriSimo01/Facebook_opt_1.3b_Quantized,21.78,22.70,25.04,23.12,0.00,59.67,0.15,pretrained,...,Original,float16,False,mit,1.30,0.0,False,7ef72ccee9d91d06967809e4e63ffbef62a9ad4a,False,False
2988,🟢,team-lucid/mptk-1b,20.84,22.70,25.48,27.11,0.00,49.72,0.00,pretrained,...,Original,float16,False,apache-2.0,1.31,0.0,True,aea467410ae0cead4fded6b98a3575e92b22862f,False,False


In [48]:
for model_name in df["Model"]:
    if "meta" in model_name in model_name:
        print(model_name)
        print(df[df["Model"] == model_name])
        # break


meta-llama/Llama-2-70b-hf
     T                      Model  Average ⬆️    ARC  HellaSwag   MMLU  \
498  🟢  meta-llama/Llama-2-70b-hf       67.87  67.32      87.33  69.83   

     TruthfulQA  Winogrande  GSM8K        Type  ... Weight type Precision  \
498       44.92       83.74  54.06  pretrained  ...    Original   float16   

    Merged  Hub License #Params (B)  Hub ❤️  Available on the hub  \
498  False          NaN       68.98   733.0                 False   

                                    Model sha Flagged    MoE  
498  ed7b07231238f836b99bf45701b9a0063576b194   False  False  

[1 rows x 21 columns]
meta-math/MetaMath-70B-V1.0
     T                        Model  Average ⬆️   ARC  HellaSwag   MMLU  \
568  🔶  meta-math/MetaMath-70B-V1.0       67.02  68.0      86.85  69.31   

     TruthfulQA  Winogrande  GSM8K        Type  ... Weight type Precision  \
568       50.98       82.32  44.66  fine-tuned  ...    Original   float16   

    Merged  Hub License #Params (B)  Hub ❤️  Ava

## Extract model info

In [77]:
model_df = read_per_model_info(
    model_id="meta-llama/Llama-2-13b-hf",
    timestamp="latest",
    scenario="harness_arc_challenge_25",
    cache_dir=CACHE_DIR
)

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


In [73]:
compute_accuracy(model_df)

Accuracy (number of acc == 1): 629 / 1172
Accuracy (number of acc == 1): 0.5366894197952219
Accuracy (number of norm_acc == 1): 681 / 1172
Accuracy (number of norm_acc == 1): 0.5810580204778157


In [80]:
model_df_0819 = read_per_model_info(
    model_id="meta-llama/Llama-2-13b-hf",
    timestamp='2023_08_19T22_35_38.117975',
    scenario="harness_arc_challenge_25",
    cache_dir=CACHE_DIR
)
compute_accuracy(model_df_0819)

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])
Accuracy (number of acc == 1): 644 / 1172
Accuracy (number of acc == 1): 0.5494880546075085
Accuracy (number of norm_acc == 1): 696 / 1172
Accuracy (number of norm_acc == 1): 0.5938566552901023


## Debug

In [1]:
# open mmlu_ordered
import pickle

with open("/home/oh/arubinstein17/github/disco-public/data/model_outputs.pkl", "rb") as f:
    model_outputs = pickle.load(f)


In [6]:
model_outputs["data"]['harness_arc_challenge_25'].keys()

dict_keys(['correctness', 'predictions'])

In [8]:
import pickle

with open("/home/oh/arubinstein17/github/disco-public/data/mmlu_fields_original.pickle", "rb") as f:
    mmlu_fields_original = pickle.load(f)


/tmp/ipykernel_3113572/1522820180.py:4: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  mmlu_fields_original = pickle.load(f)


In [9]:
print(mmlu_fields_original["data"]["harness_arc_challenge_25"].keys())

dict_keys(['correctness'])


In [11]:
import pickle

with open("/home/oh/arubinstein17/github/efficbench/data/lb_original.pickle", "rb") as f:
    lb_original = pickle.load(f)


/tmp/ipykernel_3113572/4157700953.py:4: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  lb_original = pickle.load(f)


In [12]:
print(lb_original["data"]["harness_arc_challenge_25"].keys())

dict_keys(['correctness'])


In [14]:
import pickle

with open("/home/oh/arubinstein17/github/efficbench/data/leaderboard_fields_raw_22042025.pickle", "rb") as f:
    leaderboard_fields_raw = pickle.load(f)


In [20]:
print(leaderboard_fields_raw['open-llm-leaderboard/details_logicker__SkkuDataScienceGlobal-10.7b']['harness_arc_challenge_25']['example'])

['Question: Cities control the amount of pollution that is allowed to come from cars. How does this most likely help people?\nAnswer:', 'Question: Which statement correctly describes a physical characteristic of the Moon?\nAnswer:', 'Question: Which object in the solar system is orbited by a belt of asteroids?\nAnswer:', 'Question: Light waves that cross from an air medium to a water medium will\nAnswer:', 'Question: How many valence electrons does selenium have?\nAnswer:', 'Question: As a warm moist air mass moving northward collides with a strong cold air mass moving southward, what observations will most likely be made?\nAnswer:', 'Question: Chemical weathering occurs when minerals in rocks are changed chemically. Which of these will most likely change the rate of chemical weathering on a rock?\nAnswer:', 'Question: A toy truck rolls over a smooth surface. If the surface is covered with sand, the truck will most likely roll\nAnswer:', 'Question: A town in northern Arkansas experienc

In [26]:
for key, value in leaderboard_fields_raw.items():
    if "llama" in key:
        print(key)
        print(value['harness_arc_challenge_25']['example'])
        print(value['harness_arc_challenge_25'].keys())
        print(value['harness_arc_challenge_25']['predictions'])
        break


open-llm-leaderboard/details_rameshm__llama-2-13b-mathgpt-v4
['Question: Cities control the amount of pollution that is allowed to come from cars. How does this most likely help people?\nAnswer:', 'Question: Which statement correctly describes a physical characteristic of the Moon?\nAnswer:', 'Question: Which object in the solar system is orbited by a belt of asteroids?\nAnswer:', 'Question: Light waves that cross from an air medium to a water medium will\nAnswer:', 'Question: How many valence electrons does selenium have?\nAnswer:', 'Question: As a warm moist air mass moving northward collides with a strong cold air mass moving southward, what observations will most likely be made?\nAnswer:', 'Question: Chemical weathering occurs when minerals in rocks are changed chemically. Which of these will most likely change the rate of chemical weathering on a rock?\nAnswer:', 'Question: A toy truck rolls over a smooth surface. If the surface is covered with sand, the truck will most likely rol

In [ ]:
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import pickle
import os
import argparse
import requests
import json
import sys


# sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(__file__))))
# # from utils import dict_to_h5
# from utils import dump_pickle

# sys.path.pop(0)

# CACHE_DIR = "./cache_dir"

# model_id = "logicker/SkkuDataScienceGlobal-10.7b"
model_id = "meta-llama/Llama-2-13b-hf"
creator, model = tuple(model_id.split("/"))
model_details_id = "open-llm-leaderboard/details_{:}__{:}".format(creator, model)

# s = "harness_hendrycksTest_abstract_algebra_5"
s = "harness_arc_challenge_25"
aux = load_dataset(model_details_id, s, cache_dir=CACHE_DIR)

Generating 2023_08_19T22_35_38.117975 split: 1172 examples [00:00, 9207.87 examples/s]
Generating 2023_08_23T17_28_00.015478 split: 1172 examples [00:00, 8913.82 examples/s]
Generating 2023_08_29T22_26_02.660247 split: 1172 examples [00:00, 9917.39 examples/s]
Generating latest split: 1172 examples [00:00, 9708.75 examples/s]


In [ ]:
print(aux.keys())
print(aux['latest'])
# print(aux['latest']['metrics'])
print(aux['latest']['predictions'])
print(aux['latest']['example'])
print(aux['latest']['full_prompt'])
# print(aux['latest']['hashes'])
# print(aux['latest']['dates'])
# print(aux['latest']['metrics'])
# print(aux['latest']['predictions'])
# print(aux['latest']['example'])
# print(aux['latest']['correctness'])

dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])
Dataset({
    features: ['acc', 'acc_norm', 'choices', 'cont_tokens', 'example', 'full_prompt', 'gold', 'hashes', 'input_tokens', 'instruction', 'num_asked_few_shots', 'num_effective_few_shots', 'padded', 'predictions', 'query', 'truncated'],
    num_rows: 1172
})
Column([[-10.685829162597656, -16.114404678344727, -23.640953063964844, -18.723047256469727], [-17.894737243652344, -10.447176933288574, -17.52637481689453, -21.05008888244629], [-5.0206379890441895, -3.2922346591949463, -4.592055320739746, -9.404555320739746], [-16.42238998413086, -17.457504272460938, -14.121180534362793, -10.359354972839355], [-5.114302635192871, -5.442427635192871, -0.676802396774292, -1.848677396774292]])
Column(['Question: Cities control the amount of pollution that is allowed to come from cars. How does this most likely help people?\nAnswer:', 'Question: Which statement correctly describes a p

In [75]:
model_df = read_per_model_info(
    model_id="meta-llama/Llama-2-13b-hf",
    timestamp="latest",
    scenario="harness_arc_challenge_25",
    cache_dir=CACHE_DIR
)

Available timestamps:
dict_keys(['2023_08_19T22_35_38.117975', '2023_08_23T17_28_00.015478', '2023_08_29T22_26_02.660247', 'latest'])


In [76]:
compute_accuracy(model_df)

Accuracy (number of acc == 1): 629 / 1172
Accuracy (number of acc == 1): 0.5366894197952219
Accuracy (number of norm_acc == 1): 681 / 1172
Accuracy (number of norm_acc == 1): 0.5810580204778157


In [ ]:
model_df = read_per_model_info(
    model_id="meta-llama/Llama-2-13b-hf",
    timestamp="latest",
    scenario="harness_arc_challenge_25",
    cache_dir=CACHE_DIR
)

In [ ]:
# Compute accuracy as number of rows where df["metrics"]["acc"] == 1
# Each element in df["metrics"] is likely a dict with an 'acc' key
if "metrics" in df.columns:
    num_correct = sum(1 for metric in df["metrics"] if metric.get('acc') == 1)

else:
    num_correct = sum(1 for acc in df["acc"] if acc == 1)

print(f"Accuracy (number of acc == 1): {num_correct} / {len(df)}")
print(f"Accuracy (number of acc == 1): {num_correct / len(df)}")


Accuracy (number of acc == 1): 629 / 1172
Accuracy (number of acc == 1): 0.5366894197952219


In [ ]:
# Compute accuracy as number of rows where df["metrics"]["acc"] == 1
# Each element in df["metrics"] is likely a dict with an 'acc' key
if "metrics" in df.columns:
    num_correct = sum(1 for metric in df["metrics"] if metric.get('norm_acc') == 1)

else:
    num_correct = sum(1 for acc in df["acc_norm"] if acc == 1)

print(f"Accuracy (number of norm_acc == 1): {num_correct} / {len(df)}")
print(f"Accuracy (number of norm_acc == 1): {num_correct / len(df)}")

Accuracy (number of norm_acc == 1): 681 / 1172
Accuracy (number of norm_acc == 1): 0.5810580204778157
